### Problem 06: LangGraph를 활용한 번역 및 검수 순환(Loop) 에이전트
문제 설명
영어를 한국어로 번역하는 시스템을 만듭니다. translator가 번역을 수행하면, reviewer가 번역된 결과를 확인합니다.

만약 오역이 발견되면 상태를 **'REJECTED(거절)'**로 바꾸고 다시 번역 노드로 돌려보냅니다.

검수를 통과하여 상태가 **'APPROVED(승인)'**가 되면 작업을 종료(END)하는 순환(Loop) 워크플로우를 구현하세요.

요구사항:

StateGraph를 생성하세요.

translator와 reviewer 함수를 각각 **노드(Node)**로 추가하세요. (이름은 "translator_node", "reviewer_node"로 지정)

START에서 "translator_node"로 가는 엣지를 연결하세요.

"translator_node"가 끝나면 무조건 "reviewer_node"로 가도록 일반 엣지(add_edge)를 연결하세요.

"reviewer_node"가 끝난 후에는 route_review 함수를 사용하여 **조건부 엣지(Conditional Edges)**를 연결하세요. (APPROVED면 END로, REJECTED면 다시 "translator_node"로 돌아가도록 길을 나눕니다.)

그래프를 컴파일(compile())하여 실행해 보세요!

In [4]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 1. 상태(State) 정의
class TranslationState(TypedDict):
    source_text: str   # 원본 텍스트
    translation: str   # 번역된 텍스트
    status: str        # 검수 결과 (APPROVED 또는 REJECTED)

# --- 노드 및 라우팅 함수 (수정하지 마세요) ---
def translator(state: TranslationState):
    """번역을 수행하는 노드"""
    # 만약 검수자에게 한번 빠꾸(REJECTED)를 맞았다면, 제대로 수정해서 번역함
    if state.get("status") == "REJECTED":
        print("[Translator] 지적을 반영하여 '사과'로 재번역합니다.")
        return {"translation": "사과는 맛있습니다."}
    
    # 첫 번역 (일부러 직역/오역을 함)
    print("[Translator] 첫 번역을 수행합니다.")
    return {"translation": "애플은 맛있습니다."}

def reviewer(state: TranslationState):
    """번역 결과를 검수하는 노드"""
    translation = state.get("translation", "")
    print(f"[Reviewer] 번역 결과 확인 중... (현재 번역: {translation})")
    
    # 번역에 '애플'이라는 단어가 있으면 오역으로 간주하고 거절
    if "애플" in translation:
        print("[Reviewer] 오역 발견! 다시 번역하세요. (REJECTED)")
        return {"status": "REJECTED"}
    else:
        print("[Reviewer] 번역 완벽합니다! (APPROVED)")
        return {"status": "APPROVED"}

def route_review(state: TranslationState):
    """검수 상태에 따라 갈림길을 정해주는 함수"""
    if state["status"] == "APPROVED":
        return END                # 승인되면 종료!
    else:
        return "translator_node"  # 거절되면 다시 번역하러 돌아감! (Loop)
# -------------------------------------------------------------

# 1. StateGraph 생성 (TranslationState 사용)
# 여기에 코드를 작성하세요.
graph = StateGraph(TranslationState)

# 2. 노드(Node) 추가하기 ("translator_node", "reviewer_node")
# 여기에 코드를 작성하세요.
graph.add_node("translator_node", translator)
graph.add_node("reviewer_node", reviewer)

# 3. 엣지(Edge) 연결하기
# 1) START -> translator_node (시작점 연결)
# 여기에 코드를 작성하세요.
graph.add_edge(START, "translator_node")

# 2) translator_node -> reviewer_node (번역 후엔 무조건 검수자에게 보냄)
# 여기에 코드를 작성하세요.
graph.add_edge("translator_node", "reviewer_node")
# 3) 조건부 엣지: reviewer_node -> route_review의 판단에 따라 이동 (END 또는 다시 translator_node)
# 여기에 코드를 작성하세요.
graph.add_conditional_edges("reviewer_node", route_review)

# 4. 그래프 컴파일 (조립 완료)
# 여기에 코드를 작성하세요. (결과를 app 변수에 저장하세요)
app = graph.compile()

# --- 테스트 코드 (수정하지 말고 그대로 실행해 보세요) ---
print("--- 번역/검수 에이전트 실행 시작 ---\\n")

# recursion_limit은 그래프가 뱅글뱅글 무한 루프에 빠지는 걸 막아주는 안전장치입니다.
result = app.invoke({"source_text": "Apple is delicious.", "status": ""}, {"recursion_limit": 5})

print("\\n--- 최종 완료 상태 ---")
print(result)

--- 번역/검수 에이전트 실행 시작 ---\n
[Translator] 첫 번역을 수행합니다.
[Reviewer] 번역 결과 확인 중... (현재 번역: 애플은 맛있습니다.)
[Reviewer] 오역 발견! 다시 번역하세요. (REJECTED)
[Translator] 지적을 반영하여 '사과'로 재번역합니다.
[Reviewer] 번역 결과 확인 중... (현재 번역: 사과는 맛있습니다.)
[Reviewer] 번역 완벽합니다! (APPROVED)
\n--- 최종 완료 상태 ---
{'source_text': 'Apple is delicious.', 'translation': '사과는 맛있습니다.', 'status': 'APPROVED'}
